<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 4: Data Structures and Data Types

This notebook accompanies Chapter 4 of *Python Polars: The Definitive Guide* by Jeroen Janssens and Thijs Nieuwdorp. It is adapted for **Polars 2.1** and **Google Colab**.

Before you can transform data, you need to know how Polars *stores* it. This chapter covers the containers that hold data (**Series**, **DataFrame**, **LazyFrame**) and the **data types** of the values inside them. Choosing the right data type affects correctness, memory use, and speed.

By the end of this notebook you will be able to:

- tell a Series, a DataFrame, and a LazyFrame apart, and inspect each of them
- choose a suitable numeric, temporal, textual, nested, or other data type
- explain how Polars represents **missing values** (`null`) and how that differs from `NaN`
- fill missing values with a value, a strategy, an expression, or interpolation
- **cast** data between types safely, and know what happens when a cast doesn't fit

Besides the chapter's own examples, this notebook includes a short **tour of extra data types** (Decimal, Datetime, Enum, and Object) that the chapter only lists in a table.

**How to use this notebook:** run the cells from top to bottom. A few cells *intentionally* trigger an error to teach a point; those errors are caught and printed, so they won't stop the notebook.

## 0. Setup

We pin Polars to the **2.1 series** so that everyone sees the same behavior. The `graph` extra installs what Polars needs to draw query plans (used once, for the LazyFrame). We use `%pip` so the package is installed into the running notebook kernel.

In [ ]:
%pip install -q "polars[graph]==2.1.*"

### Imports

In [ ]:
from datetime import date, datetime, time, timedelta
from decimal import Decimal
from pathlib import Path
import struct

import numpy as np
import polars as pl
import polars.selectors as cs


print(f"Polars {pl.__version__}")


def try_it(func):
    """Run func(); print its result, or the error it raises, without stopping the notebook."""
    try:
        result = func()
    except Exception as error:
        print(f"{type(error).__name__}: {error}")
    else:
        print("No error raised. Result:")
        print(result)

### Download the data

We need one small file, `fruit.csv`. The `curl` flags `-fsSL` make the download fail loudly on HTTP errors, instead of silently saving an error page as if it were data. Defining the path once avoids repeating it.

In [ ]:
%%bash
set -euo pipefail

curl -fsSL -o sample_data/fruit.csv \
    https://raw.githubusercontent.com/davidrpugh/data-analytics-with-polars/refs/heads/main/data/fruit.csv

In [ ]:
FRUIT_CSV = Path("sample_data/fruit.csv")

print(FRUIT_CSV.read_text())

## Series, DataFrames, and LazyFrames

Polars stores data in three kinds of containers:

| Container | What it is | Where is the data? |
|---|---|---|
| **Series** | A one-dimensional, *named* sequence of values that all share one data type | In memory |
| **DataFrame** | A two-dimensional table, internally a list of equally long Series (its columns) | In memory |
| **LazyFrame** | A *plan* for producing a DataFrame | Nowhere yet: only instructions |

### Series

A Series has a **name**, a **data type**, and a **length**. All values must share one data type; Polars infers it from the values you pass (here: floats, so `Float64`).

In [ ]:
sales_series = pl.Series("sales", [150.00, 300.00, 250.00])
sales_series

In [ ]:
print(f"name:  {sales_series.name}")
print(f"dtype: {sales_series.dtype}")
print(f"len:   {len(sales_series)}")

### DataFrame

A DataFrame is a table whose columns are Series of the **same length**. A column's name is its Series name. Here we build one from a *dictionary*, mixing an existing Series with a plain Python list. Polars infers `customer_id` as `Int64`.

In [ ]:
sales_df = pl.DataFrame(
    {
        "sales": sales_series,
        "customer_id": [24, 25, 26],
    }
)

sales_df

Polars gives you several ways to **inspect** a DataFrame without printing it all. These are worth knowing from day one:

- `shape`, `height`, `width`: the dimensions
- `schema`: the column names mapped to their data types (also available as `columns` and `dtypes`)
- `null_count()`: missing values per column
- `describe()`: summary statistics
- `glimpse()`: a compact, transposed preview that works well for wide tables
- `estimated_size()`: approximate memory use

In [ ]:
print(f"shape:  {sales_df.shape}  (height={sales_df.height}, width={sales_df.width})")
print(f"schema: {sales_df.schema}")
print(f"size:   {sales_df.estimated_size('b')} bytes")

In [ ]:
sales_df.describe()

In [ ]:
sales_df.glimpse()

#### Other ways to build a DataFrame

The same table can be created from a list of Series or from a list of row dictionaries. Which you choose depends on how your data arrives.

You can also control types explicitly. `schema=` sets the type of the columns you list, and `schema_overrides=` changes only some of them while inferring the rest.

In [ ]:
from_series = pl.DataFrame([sales_series, pl.Series("customer_id", [24, 25, 26])])

from_rows = pl.DataFrame(
    [
        {"sales": 150.0, "customer_id": 24},
        {"sales": 300.0, "customer_id": 25},
        {"sales": 250.0, "customer_id": 26},
    ]
)

assert from_series.equals(sales_df) and from_rows.equals(sales_df)
print("All three constructions produce the same DataFrame.")

In [ ]:
pl.DataFrame({"customer_id": [24, 25, 26]}, schema_overrides={"customer_id": pl.UInt16})

### LazyFrame

A **LazyFrame** looks like a DataFrame but holds **no data**. It holds a *plan*: a list of instructions describing how to read and transform data. Nothing is read or computed until you ask for the result with `.collect()`. Because Polars sees the whole plan first, it can optimize it, for example by reading only the columns you actually use.

We start a plan with `scan_csv` (the lazy counterpart of `read_csv`) and add a column that flags heavy fruit. Displaying the LazyFrame shows the plan, not data.

By convention LazyFrame variables are often named `lf`.

In [ ]:
lf = pl.scan_csv(FRUIT_CSV).with_columns(is_heavy=pl.col("weight") > 200)
lf

A query plan is read **from the bottom up**: first the CSV is scanned, then the new column is added. `explain()` prints the plan as text and needs no extra tools.

In [ ]:
print(lf.explain())

The plan can also be drawn as a diagram with `show_graph()`.

In [ ]:
lf.show_graph()


A LazyFrame **knows its column types without reading any data**. Use `collect_schema()` to ask for them. (Prefer this over the `schema` or `columns` attributes on a LazyFrame: resolving the schema can require work, and `collect_schema()` makes that explicit.)

Then `collect()` executes the plan and returns a regular DataFrame.

In [ ]:
print(lf.collect_schema())

In [ ]:
lf.collect()

#### Moving between lazy and eager

You can start lazy from an existing DataFrame with `df.lazy()` and finish with `collect()`. The type changes at each step:

In [ ]:
lazy_sales = sales_df.lazy()
print(f"{type(sales_df).__name__} -> {type(lazy_sales).__name__}")

In [ ]:
collected = lazy_sales.select("customer_id", double_sales=pl.col("sales") * 2).collect()
print(f"{type(lazy_sales).__name__} -> {type(collected).__name__}")

In [ ]:
collected

We cover when to use each mode in Chapter 5.

## Data Types

Polars stores data using the **Apache Arrow** columnar memory format (Chapter 18 explains why). Many data types come in several **bit sizes**, so you can choose the smallest type that fits your data and use less memory.

Here is a map of the data types, grouped as in the chapter:

| Group | Types |
|---|---|
| Numeric | `Int8`, `Int16`, `Int32`, `Int64`, `UInt8`, `UInt16`, `UInt32`, `UInt64`, `Float32`, `Float64`, `Decimal` |
| Temporal | `Date`, `Datetime`, `Duration`, `Time` |
| Nested | `Array`, `List`, `Struct` |
| Text | `String`, `Categorical`, `Enum` |
| Other | `Boolean`, `Binary`, `Null`, `Object` |

The rest of this section tours these types.

### Numeric types

When you don't say otherwise, Polars infers `Int64` for whole numbers and `Float64` for decimals.

In [ ]:
print(pl.Series([1, 2, 3]).dtype)
print(pl.Series([1.0, 2.0, 3.0]).dtype)

Mixing integers and floats in one column gives raises an error (unless you set `strict=False`):

In [ ]:
print(pl.Series([1, 2.5, 3]).dtype)

Smaller types use less memory but can only hold a limited range. The integer ranges below come from NumPy's `iinfo`, which matches the ranges of the corresponding Polars types.

**Signed** types (`Int*`) can hold negative numbers; **unsigned** types (`UInt*`) cannot but reach twice as high.

In [ ]:
integer_types = {
    "Int8": np.int8, "Int16": np.int16, "Int32": np.int32, "Int64": np.int64,
    "UInt8": np.uint8, "UInt16": np.uint16, "UInt32": np.uint32, "UInt64": np.uint64,
}

pl.DataFrame(
    {
        "type": list(integer_types),
        "bits": [np.iinfo(t).bits for t in integer_types.values()],
        "min": [f"{np.iinfo(t).min:,}" for t in integer_types.values()],
        "max": [f"{np.iinfo(t).max:,}" for t in integer_types.values()],
    }
)

You choose a type with the `dtype` argument of the Series constructor, or later with `cast` (covered below):

In [ ]:
pl.Series("small", [1, 2, 3], dtype=pl.UInt8)

#### Floats: `Float32` vs. `Float64`

Floating-point numbers trade **precision** for range and size. `Float64` (the default) uses 64 bits; `Float32` uses 32 bits, which halves the memory but keeps fewer significant digits. Compare how each stores one third:

In [ ]:
print("Float64:", pl.Series([1 / 3], dtype=pl.Float64).to_list()[0])
print("Float32:", pl.Series([1 / 3], dtype=pl.Float32).to_list()[0])

Floats cannot represent every decimal number exactly, which leads to classic surprises:

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

**How a `Float32` is stored.** Its 32 bits are split into a *sign* bit, an 8-bit *exponent*, and a 23-bit *fraction*. The value is `(-1)^sign × (1 + fraction) × 2^(exponent − 127)`, where 127 is the "bias".

In [ ]:
def float32_parts(x: float) -> tuple[str, str, str]:
    """Return the sign, exponent, and fraction bits of x stored as a 32-bit float."""
    (as_int,) = struct.unpack(">I", struct.pack(">f", x))
    bits = format(as_int, "032b")
    return bits[0], bits[1:9], bits[9:]


sign, exponent, fraction = float32_parts(13.0)
print(sign, exponent, fraction)

value = (-1) ** int(sign) * (1 + int(fraction, 2) / 2**23) * 2 ** (int(exponent, 2) - 127)
assert (sign, exponent, fraction) == ("0", "10000010", "10100000000000000000000")
assert value == 13.0
print(f"exponent = {int(exponent, 2)}, value = {value}")

#### Decimal: exact base-10 arithmetic

When you need **exact** decimal results, as with money, use `Decimal` instead of floats. You choose the *precision* (total number of digits) and the *scale* (digits after the decimal point). Python's `decimal.Decimal` values are converted directly.

Check the Polars documentation for the current status of `Decimal`: parts of it have been marked as unstable in some releases. The guarded cells print whatever your version does.

In [ ]:
prices = pl.Series("price", [Decimal("19.99"), Decimal("0.10"), Decimal("0.20")], dtype=pl.Decimal(10, 2))

In [ ]:
prices

In [ ]:
print(prices.dtype)

In [ ]:
prices.sum()

### Temporal types

Four types handle time:

| Type | Meaning | Stored as |
|---|---|---|
| `Date` | A calendar date | Days since 1970-01-01 (the Unix epoch) |
| `Datetime` | A date and a time of day, with a **time unit** (default microseconds) and an optional **time zone** | Time since the epoch |
| `Time` | A time of day | Nanoseconds since midnight |
| `Duration` | A length of time (the difference between two points) | A count in a time unit |

They are built from Python's `datetime` objects:

In [ ]:
temporal_df = pl.DataFrame(
    {
        "date": [date(2024, 3, 1)],
        "datetime": [datetime(2024, 3, 1, 12, 30)],
        "time": [time(12, 30)],
        "duration": [timedelta(hours=2, minutes=15)],
    }
)

temporal_df

A `Date` really is just a day count. `to_physical()` reveals the underlying number (one day after the epoch is `1`):

In [ ]:
pl.Series("d", [date(1970, 1, 1), date(1970, 1, 2), date(2024, 3, 1)]).to_physical()

**Datetimes and time zones.** A plain `Datetime` has *no* time zone (it is "naive"). That is fine for local timestamps, but for data that crosses regions, attach a time zone. `replace_time_zone` *labels* the existing wall-clock time with a zone; `convert_time_zone` *translates* a zone-aware timestamp to another zone. You can also pick a coarser or finer **time unit** with `pl.Datetime("ms")`, `"us"`, or `"ns"`.

Subtracting two Datetimes gives a `Duration`.

In [ ]:
stamps = pl.DataFrame({"naive": [datetime(2024, 3, 1, 12, 0)]}).with_columns(
    utc=pl.col("naive").dt.replace_time_zone("UTC"),
)
stamps = stamps.with_columns(amsterdam=pl.col("utc").dt.convert_time_zone("Europe/Amsterdam"))

In [ ]:
print(stamps.schema)

In [ ]:
stamps

In [ ]:
pl.DataFrame(
    {
        "start": [datetime(2024, 3, 1, 9, 0)],
        "end": [datetime(2024, 3, 1, 17, 30)]
    }
).with_columns(
    elapsed=pl.col("end") - pl.col("start")
)

Chapter 12 covers parsing and working with temporal data in depth.

### Text types: String, Categorical, and Enum

`String` holds text of any length (UTF-8). When a column has **few distinct values repeated many times** (sizes, countries, statuses), you can store it more efficiently as a categorical.

- **`Categorical`** learns its categories from the data. Internally each distinct string gets an integer code, and the rows store the codes.
- **`Enum`** is a categorical whose categories you **declare up front**. Values outside the list are an error, and the categories have a meaningful **order**.

Prefer `Enum` when the possible values are known in advance. It documents your assumptions, catches typos, and sorts logically. Compare:

In [ ]:
sizes = pl.Enum(["small", "medium", "large"])

sizes_series = pl.Series("size", ["large", "small", "medium", "small"], dtype=sizes)

print(sizes_series.dtype)
print("sorted:", sizes_series.sort().to_list())          # small, medium, large (category order)
print("codes: ", sizes_series.to_physical().to_list())

In [ ]:
text_series = pl.Series("size", ["large", "small", "medium", "small"])           # String
cat_series = pl.Series("size", ["large", "small", "medium", "small"], dtype=pl.Categorical)

print("String sorted:     ", text_series.sort().to_list())      # alphabetical
print("Categorical dtype: ", cat_series.dtype)

An `Enum` refuses values that aren't in its list. That strictness is the feature: it catches bad data at the door.

In [ ]:
pl.Series("size", ["small", "huge"], dtype=sizes)

### Other types: Boolean, Binary, and Null

- `Boolean` holds `True`/`False` (one bit per value) and supports missing values.
- `Binary` holds raw bytes of variable length.
- `Null` is the type of a column that contains *only* missing values. Polars can't tell what the type should be, so it uses `Null` until you cast it.

In [ ]:
print(pl.Series([True, False, None]).dtype)
print(pl.Series([b"\x00\x01", b"\xff"]).dtype)
print(pl.Series([None, None]).dtype)

### Object: a stowaway for arbitrary Python objects

The `Object` type wraps **any Python object**, for example a trained machine learning model. Polars doesn't know what's inside an Object column, so:

- it can carry the objects along (for instance through filters and sorts on *other* columns), but
- it **can't compute on them** or optimize anything involving them.

Think of an Object column as a *passenger* in the DataFrame. Use it only when no other data type can represent your data. Often a `Struct`, `List`, or a few plain columns can.

In [ ]:
class Model:
    """A stand-in for something Polars has no data type for, such as a trained model."""

    def __init__(self, name: str):
        self.name = name

    def __repr__(self) -> str:
        return f"Model({self.name!r})"


objects_df = pl.DataFrame(
    {
        "id": [1, 2, 3],
        "model": pl.Series("model", [Model("a"), Model("b"), Model("c")], dtype=pl.Object),
    }
)

print(objects_df.schema)

In [ ]:
objects_df.filter(pl.col("id") > 1)    # the Object column rides along

But Polars' own operations don't understand it. Asking for the length of the "strings" in an Object column fails, because they aren't strings. The objects are still ordinary Python objects once you pull them out:

In [ ]:
objects_df.select(pl.col("model").str.len_chars())

In [ ]:
objects_df.get_column("model").to_list()[0].name

### Nested data types: Array, List, and Struct

Nested types store several values in **one cell**.

| Type | Holds | Length |
|---|---|---|
| `Array` | Values of **one** type | **Fixed**, the same for every row |
| `List` | Values of **one** type | **Variable** per row |
| `Struct` | Several **named fields**, each with its own type | Fixed set of fields |

#### Array

Use an `Array` for fixed-size data such as coordinates or image pixels. Because the length is known, it is more compact and faster than a `List`. You give the inner type and the shape.

In [ ]:
coordinates = pl.DataFrame(
    [
        pl.Series("point_2d", [[1, 3], [2, 5]], dtype=pl.Array(inner=pl.Int64, shape=2)),
        pl.Series("point_3d", [[1, 7, 3], [8, 1, 0]], dtype=pl.Array(inner=pl.Int64, shape=3)),
    ]
)

In [ ]:
print(coordinates.schema)

In [ ]:
coordinates

Every row of an `Array` column must have the length you declared. A ragged input is rejected:

In [ ]:
pl.Series("bad", [[1, 2], [3]], dtype=pl.Array(inner=pl.Int64, shape=2))

#### List

A `List` allows a *different length in each row*. Note that this is not Python's `list`, which can mix types: a Polars List holds one inner type. Here the inner types (`Float64` and `Int64`) are inferred. `list.len()` counts the elements per row.

In [ ]:
weather_readings = pl.DataFrame(
    {
        "temperature": [[72.5, 75.0, 77.3], [68.0, 70.2]],
        "wind_speed": [[15, 20], [10, 12, 14, 16]],
    }
)


In [ ]:
print(weather_readings.schema)

In [ ]:
weather_readings.with_columns(n_temperatures=pl.col("temperature").list.len())

#### Struct

A `Struct` bundles several **named fields** into one column, like a Python dictionary per row. Field names follow the same convention as column names; we use `snake_case`. A Struct is how Polars lets one expression work on several columns at once.

In [ ]:
rating_series = pl.Series(
    "ratings",
    [
        {"movie": "Cars", "theatre": "NE", "avg_rating": 4.5},
        {"movie": "Toy Story", "theatre": "ME", "avg_rating": 4.9},
    ],
)

In [ ]:
print(rating_series.dtype)

In [ ]:
rating_series

You can list the fields and expand ("unnest") the Struct into ordinary columns:

In [ ]:
print(rating_series.struct.fields)

In [ ]:
rating_series.struct.unnest()

To control the field types yourself, declare the Struct explicitly:

In [ ]:
rating_dtype = pl.Struct({"movie": pl.String, "theatre": pl.String, "avg_rating": pl.Float32})

pl.Series("ratings", [{"movie": "Cars", "theatre": "NE", "avg_rating": 4.5}], dtype=rating_dtype).dtype

We cover working with `List`, `Array`, and `Struct` in Chapter 12.

## Missing Values

In Polars, missing data is **always `null`**, for **every** data type, including integers and Booleans. Polars tracks which values are missing in a separate **validity bitmap** (one bit per value), so checking for or counting nulls is very cheap: `null_count()` and `is_null()` don't have to inspect the values.

Let's create a column with some missing values. Note that it stays an **integer** column: unlike pandas, a missing value doesn't turn integers into floats.

In [ ]:
missing_df = pl.DataFrame(
    {
        "value": [None, 2, 3, 4, None, None, 7, 8, 9, None],
    },
)

In [ ]:
print(missing_df.schema)

In [ ]:
print(missing_df.null_count())

In [ ]:
missing_df

### `NaN` is not `null`

`NaN` ("not a number") is a special **floating-point** value, produced by operations like `0.0 / 0.0`. In Polars it is a regular value, **not** a missing value. So `is_null()` and `null_count()` don't count it, and `fill_null()` doesn't fill it. Use `is_nan()` and `fill_nan()` instead:

In [ ]:
pl.DataFrame(
    {
        "x": [1.0, None, float("nan")]
    }
).select(
    "x",
    is_null=pl.col("x").is_null(),
    is_nan=pl.col("x").is_nan(),
)

### Filling in missing values

`fill_null()` can be used in four ways. Each is an *expression* applied inside `with_columns` (Chapter 7 explains expressions).

**1. A single value.**

In [ ]:
missing_df.with_columns(filled_with_single=pl.col("value").fill_null(-1))

**2. A fill strategy.** A strategy picks the replacement for you:

| Strategy | Fills with |
|---|---|
| `forward` | the previous non-null value |
| `backward` | the next non-null value |
| `min`, `max` | the smallest or largest value |
| `mean` | the mean |
| `zero`, `one` | `0` or `1` |

Look at the edges: `forward` can't fill a **leading** null (nothing comes before it) and `backward` can't fill a **trailing** null.

Also look at the `mean` column. The column is an *integer* column, so check whether the mean (5.5) was cut to an integer, as the book describes, or kept as a float in your version. Compare with the expression-based fill in the next step, which keeps the decimals.

In [ ]:
missing_df.with_columns(
    fill_forward=pl.col("value").fill_null(strategy="forward"),
    fill_backward=pl.col("value").fill_null(strategy="backward"),
    fill_min=pl.col("value").fill_null(strategy="min"),
    fill_max=pl.col("value").fill_null(strategy="max"),
    fill_mean=pl.col("value").fill_null(strategy="mean"),
    fill_zero=pl.col("value").fill_null(strategy="zero"),
    fill_one=pl.col("value").fill_null(strategy="one"),
)

**3. An expression.** You can fill with *any* expression, such as the column's own mean. This gives `5.5` and the column becomes a float, because 5.5 isn't an integer.

In [ ]:
missing_df.with_columns(
    expression_mean=pl.col("value").fill_null(pl.col("value").mean())
)

**4. Interpolation.** For ordered data (such as a time series), you can estimate missing values *between* known ones. `interpolate()` returns floats, and it can't fill the edges, because there's nothing on one side to interpolate toward.

Combining it with the strategies fixes that: first interpolate the inside, then use `backward`/`forward` for the edges. Only do this when it makes sense for your data. Interpolating unordered data is meaningless.

In [ ]:
missing_df.with_columns(
    interpolated=pl.col("value").interpolate(),
    interpolated_and_edges_filled=(
        pl.col("value").interpolate().fill_null(strategy="backward").fill_null(strategy="forward")
    ),
)

**Or just drop them.** If you can't reasonably fill a missing value, remove the rows with `drop_nulls()`. Always consider *why* values are missing before you decide how to handle them.

In [ ]:
missing_df.drop_nulls()

## Data Type Conversion

Sometimes a column arrives with the wrong type: for instance, numbers read as strings from a CSV. You convert with `cast`, either on an **expression** (`pl.col("a").cast(...)`) or on a whole **DataFrame** (`df.cast(...)`).

Using the right type is more than tidiness. The same three ids stored as text versus as a small integer:

In [ ]:
string_df = pl.DataFrame({"id": ["10000", "20000", "30000"]})
print(string_df)
print(f"Estimated size: {string_df.estimated_size('b')} bytes")

In [ ]:
int_df = string_df.select(pl.col("id").cast(pl.UInt16))
print(int_df)
print(f"Estimated size: {int_df.estimated_size('b')} bytes")

Three rows are too small to measure anything convincingly (and the size of strings depends on Polars' internal layout), so let's repeat the experiment with **a million values** and let the code report the ratio instead of hardcoding a claim. (`UInt16` holds 0 to 65,535, so our ids, which stay below 60,000, fit.)

In [ ]:
ids_int = pl.int_range(1_000_000, eager=True) % 60_000
ids_str = ids_int.cast(pl.String)
ids_u16 = ids_str.cast(pl.UInt16)

size_str = ids_str.estimated_size("mb")
size_u16 = ids_u16.estimated_size("mb")
print(f"String: {size_str:.1f} MB")
print(f"UInt16: {size_u16:.1f} MB  ({size_u16 / size_str:.0%} of the String size)")

### Casting is strict by default, and that's good

What if a value **doesn't fit** the target type? By default (`strict=True`) Polars raises an error rather than silently corrupting your data. Here `70_000` doesn't fit in a `UInt16`, and `"x"` isn't a number:

In [ ]:
pl.Series("id", [10_000, 20_000, 70_000]).cast(pl.UInt16)

In [ ]:
pl.Series("id", ["1", "x"]).cast(pl.Int64)

With `strict=False`, values that can't be converted become **`null`** instead of raising an error. This is handy for messy data, but it can *hide* problems, so **always check how many values were lost**:

In [ ]:
raw = pl.Series("id", [10_000, 20_000, 70_000])
casted = raw.cast(pl.UInt16, strict=False)

print(casted.to_list())
print(f"values lost to null: {casted.null_count() - raw.null_count()}")

Casting a float to an integer **truncates** toward zero. It does not round. Use `round()` first if you want rounding:

In [ ]:
floats = pl.Series("x", [1.9, -1.9, 2.5])
print("cast:        ", floats.cast(pl.Int64).to_list())
print("round + cast:", floats.round(0).cast(pl.Int64).to_list())

### Casting a whole DataFrame

`df.cast(...)` converts several columns at once. It accepts:

1. a **single type**, applied to every column
2. a **dictionary** of column names to types
3. a dictionary of **types to types** ("turn every Float64 into a Float32")
4. a dictionary using **column selectors** as keys (Chapter 10 covers selectors)

We'll use this DataFrame, which has an integer, a float, and a string column:

In [ ]:
data_types_df = pl.DataFrame(
    {
        "id": [10000, 20000, 30000],
        "value": [1.0, 2.0, 3.0],
        "value2": ["1", "2", "3"],
    }
)

print(data_types_df.schema)

**One type for everything.** This works here because all three columns hold values that fit in a `UInt16`, including the text, which Polars parses as a number.

In [ ]:
data_types_df.cast(pl.UInt16)

**A dictionary of column names to types**, to cast each column differently:

In [ ]:
data_types_df.cast({"id": pl.UInt16, "value": pl.Float32, "value2": pl.UInt8})

**Types to types.** Cast *all* `Float64` columns to `Float32` and *all* `String` columns to `UInt8`. The `id` column is untouched, because we didn't mention its type.

In [ ]:
data_types_df.cast({pl.Float64: pl.Float32, pl.String: pl.UInt8})

**Column selectors.** Selectors describe columns by property. Here `cs.numeric()` matches the integer and float columns, but the text column keeps its type.

In [ ]:
data_types_df.cast({cs.numeric(): pl.UInt16})

**Shrinking every column to its best-fitting type.** Selectors are also *expressions*, so you can use them with `with_columns` to give each kind of column its own target type in a single call. This is the idea behind the memory-saving example earlier.

In [ ]:
optimized = data_types_df.with_columns(
    cs.integer().cast(pl.UInt16),
    cs.float().cast(pl.Float32),
    cs.string().cast(pl.UInt8),
)

print(f"before: {data_types_df.estimated_size('b')} bytes  {data_types_df.dtypes}")
print(f"after:  {optimized.estimated_size('b')} bytes  {optimized.dtypes}")

Remember: `Float32` keeps fewer significant digits than `Float64`, so only use it when you don't need the extra precision.

Basic casting can't do everything. Some conversions need extra information, such as the format of a date string. Chapter 12 covers parsing text into temporal types.

## Type inference and strictness

When you build a Series or DataFrame without declaring types, Polars **infers** them from the data. Compatible values are combined (integers and floats become floats), but **incompatible** values are an error. Pandas would quietly fall back to the generic `object` type; Polars refuses. If you do want to mix, convert to a common type first.

In [ ]:
pl.Series("mixed", [1, "a", 2.5])

In [ ]:
pl.Series("as_text", [str(v) for v in [1, "a", 2.5]])

## Takeaways

- **Series** are named, typed columns; a **DataFrame** is a list of equally long Series; a **LazyFrame** holds only a plan, and `collect()` turns it into a DataFrame.
- Inspect data with `schema`, `shape`, `describe()`, `glimpse()`, `null_count()`, and `estimated_size()`. Use `collect_schema()` on a LazyFrame.
- Choose the **smallest type that fits** your data, but remember that casting to a smaller type can fail or lose precision.
- Use **`Decimal`** for exact decimal arithmetic, **`Enum`** for text with known categories, and **`Datetime` with a time zone** for data that spans regions.
- Use **`Object`** only as a last resort. Polars can carry the values but can't compute on them.
- **`Array`** is fixed-length, **`List`** is variable-length, and **`Struct`** bundles named fields.
- Missing data is always **`null`**, for every type. **`NaN`** is a separate, floating-point-only value.
- Fill nulls with a **value**, a **strategy**, an **expression**, or **interpolation**, or drop them. Think about *why* data is missing first.
- Casting is **strict** by default, which protects your data. If you use `strict=False`, count the resulting nulls.

### Next up

In Chapter 5 we compare the **eager** and **lazy** APIs in depth, and when to use each.

### Try it yourself


1. Create a `UInt8` Series with the values `[0, 128, 255]`, then cast it to `Int8` with and without `strict=False`. What happens, and why?

2. What is the smallest unsigned integer type that can hold the number `100_000`? Check with the table above, then confirm by casting.

3. Make an `Enum` called `grade` with categories `"C" < "B" < "A"`, build a Series with a few grades, and sort it. Then try adding a grade `"D"`.

4. Read `fruit.csv` lazily, then use `collect_schema()` to print its types without loading the data. Which columns could be given a smaller or more specific type (for example `Enum`)?

5. Use `pl.Series([1.0, None, float("nan")])` to show the difference between `null_count()` and counting `is_nan()` values.